# Heston calibration study — piecewise-constant chaos

Run 100 independent calibrations from freshly projected Heston-to-chaos coefficients. Every Heston fit starts from the deterministic heuristic computed from this Heston IV surface; each projection uses its own seed and the notebook's original 80,000-path, 101-point budget. Training seeds differ by run, and training paths refresh every 50 optimizer iterations as before. One `tqdm` bar shows the current phase and counts checkpointed calibrations. Fit and runtime summaries follow. This notebook is independent of the single-calibration and real-data notebooks.


## 1. Imports and data

Load project libraries and the calibration/test surfaces. Paths work from the repository or the notebook directory.

In [ ]:
from pathlib import Path
import importlib
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Heston/Piecewise constant"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
from wiener_chaos.experiments import calibration as study
study = importlib.reload(study)
print(f"Study workflow loaded: {study.__file__}")

DATA_PATH = ROOT / "Heston/heston_data.pkl"
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Calibration settings

Calibration and calibration study share the same effective configuration:
`P=2`, `d=2`, the seven-interval piecewise grid, derivative backend, and the same
Gauss–Hermite rules and Monte Carlo budgets. Heston starts from the surface-derived
guess and uses 20,000 × 4 projection paths, 101 simulation time points, and an
optimizer limit of 100 iterations.

Chaos calibration uses `lr=1e-4`, at most 10,000 steps, patience 30,
`tol_error=1e-7`, and `limit_error_above=1e-6`. Training paths refresh every 50
steps. Degree-2 control variates are active immediately, validation monitoring is
disabled, and both regularization weights are zero (`anchor_lambda=0`,
`reg_lambda=0`). Scheduler and stopping settings match between the two notebooks.

Printing, study run count, and GPU selection retain their notebook-specific values.
The study varies projection/training seeds across runs.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 0  # Index of the GPU used for all model computations.
if torch.cuda.is_available() and CUDA_DEVICE_INDEX >= torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    # Large thread pools can slow the many small matrix operations in this study.
    torch.set_num_threads(1)

CONFIG = {
    # Number of independent fits in the study.
    "n_calibrations": 100,
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": "derivative",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 2,
    # Time-grid endpoints defining the piecewise Brownian basis.
    "pi_M": [0.0, 0.0821, 0.1725, 0.2491, 0.4983, 0.9884, 1.4867, 1.974],
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        "projection_control": "none",
        "projection_antithetic": False,
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 20_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 4,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 101,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 100,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["Gauss_Hermite"] * 2 + ["MC"] * 5,
        # MC paths per calibration maturity; None for quadrature.
        "N": [None, None] + [100_000] * 5,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 7,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [40, 25] + [None] * 5,
    },
    # Pricing allocations at held-out test maturities.
    "pricing_test": {
        # Integration method for each held-out maturity.
        "method": ["Gauss_Hermite"] * 2 + ["MC"] * 4,
        # MC paths per held-out maturity; None for quadrature.
        "N": [None, None] + [100_000] * 4,
        # Independent MC batches per held-out maturity.
        "Ns": [1] * 6,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [30, 15] + [None] * 4,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 10_000,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 30,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 1e-7,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 1e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": None,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
OUTPUT_BASE_DIR = STUDY_DIR / "Statistics calibration" / "study"

print(f"Device: {DEVICE} | Calibrations: {CONFIG['n_calibrations']}")
print(f"Results base directory: {OUTPUT_BASE_DIR}")
maturity_table = pd.DataFrame([
    {"set": label, "maturity": t, "pricing": method}
    for label, key in (("Calibration", "calibr"), ("Test", "test"))
    for t, method in zip(data[f"maturities_{key}"], CONFIG[f"pricing_{key}"]["method"])
]).sort_values("maturity", ignore_index=True)
display(maturity_table)

# Reuse completed studies automatically; set False to run or resume calibrations.
SAVED_RESULTS_DIR = OUTPUT_BASE_DIR
LOAD_SAVED_RESULTS = (study.resolve_study_directory(SAVED_RESULTS_DIR) / "results.json").is_file()


## 3. Study workflow

Run, checkpoint, and loading functions are in [calibration.py](../../wiener_chaos/experiments/calibration.py). Data, hyperparameters, and CUDA device are set here independently.


## 4. Run or load the study

Load the latest completed study automatically when available. Otherwise, run
or resume the configured calibrations. Set `LOAD_SAVED_RESULTS=False` explicitly
to run or resume calibrations. Completed runs are checkpointed, and a completed
study becomes the default for analysis and plotting automatically.


In [ ]:
if LOAD_SAVED_RESULTS:
    if SAVED_RESULTS_DIR is None:
        raise ValueError("Set SAVED_RESULTS_DIR to a completed study folder before loading.")
    records, OUTPUT_DIR, ANALYSIS_CONFIG = study.load_synthetic_results(SAVED_RESULTS_DIR)
    print(f"Loaded {len(records)} saved calibrations from {OUTPUT_DIR}")
else:
    with tqdm(total=CONFIG["n_calibrations"], desc="Heston calibrations", unit="calibration") as progress_bar:
        records, OUTPUT_DIR = study.run_study(
            data, CONFIG, OUTPUT_BASE_DIR, DATA_PATH,
            progress=study.tqdm_progress_callback(progress_bar),
        )
    ANALYSIS_CONFIG = CONFIG
    print(f"Completed {len(records)} calibrations. Results saved in: {OUTPUT_DIR}")
runs = study.results_tables(records)


## 5. Fit quality and runtime

Compare implied-volatility errors on calibration and held-out test maturities. Initialization time includes each Heston fit and chaos projection; calibration time covers the chaos optimizer; total is their sum. Held-out pricing is outside that sum. Training and test seeds differ by run.


In [ ]:
run_table = runs.set_index("run").copy()
run_table["initialization_s"] = run_table["elapsed_seconds_init"]
run_table["calibration_s"] = run_table["elapsed_seconds_chaos_calibration"]
run_table["total_s"] = run_table["initialization_s"] + run_table["calibration_s"]
quality_columns = ["iv_mae_calibration_bp", "iv_mae_test_bp", "loss_ins", "loss_oos",
                   "invalid_iv_calibration", "invalid_iv_test",
]
fit_summary = run_table[quality_columns].agg(["count", "mean", "std", "min", "max"]).T
timing_summary = run_table[["initialization_s", "calibration_s", "total_s"]].agg(
    ["count", "mean", "std", "min", "max"]
).T
display(fit_summary)
display(timing_summary)
print(f"Sum of initialization times: {run_table['initialization_s'].sum():.2f}s | "
      f"Sum of calibration times: {run_table['calibration_s'].sum():.2f}s | "
      f"Combined: {run_table['total_s'].sum():.2f}s")
print(f"Runs with invalid calibration IVs: {int((run_table['invalid_iv_calibration'] > 0).sum())}")

In [ ]:
figures = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, column in (("Calibration", "iv_mae_calibration_bp"), ("Test", "iv_mae_test_bp")):
    axes[0].plot(run_table.index, run_table[column], ".", label=label)
axes[0].set(xlabel="Calibration run", ylabel="IV MAE (bp)", title="Fit quality across runs")
axes[0].legend()
run_table[["initialization_s", "calibration_s", "total_s"]].plot(ax=axes[1])
axes[1].set(xlabel="Calibration run", ylabel="Seconds", title="Initialization and calibration time")
fig.tight_layout()
plt.show()
figures["study_summary"] = fig

## 6. Saved results

The study folder contains `manifest.json` and aggregate `results.json`, with
one checkpoint per calibration in `runs/run_XXX/result.json`. Coefficient tensors
are saved in `coefficients/`; fit and timing tables in `tables/`; summary plots
in `figures/`. Additional coefficient diagnostics are grouped in `diagnostics/`.
The relative `study_latest.json` pointer selects the completed study for loading
and plotting automatically.


In [ ]:
study.save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": torch.tensor([r["initial_coefficients"] for r in records], dtype=torch.float32),
                  "calibrated_coefficients": torch.tensor([r["d_a"] for r in records], dtype=torch.float32)},
    tables={"calibrations": run_table, "fit_summary": fit_summary, "timing_summary": timing_summary},
    figures=figures,
)
print(f"Study results saved in: {OUTPUT_DIR}")
